# inference-mode-step — faded example 1: Fill the inference_mode decorator on step

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `inference-mode-step`. Running the beacon reports progress on the `PyTorch: Inference mode step` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Inference mode step` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`inference-mode-step`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "inference-mode-step"
DD_SUBTOPIC = "PyTorch: Inference mode step"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A hand-rolled optimizer's `step` must run with autograd disabled so the bare in-place leaf update `p -= lr * p.grad` is legal. The `@t.inference_mode()` decorator on `step` provides that.

## Faded exercise 1

Implement `InferenceSGD`. The body of `step` does the bare in-place update for each param with a gradient. Add the decorator that makes this legal (do not switch to `p.data`). Complete the blanked decorator on `step`.

**Fill in:** the @t.inference_mode() decorator applied to the step method

In [ ]:
import torch as t

t.manual_seed(3)

class InferenceSGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    @t.inference_mode()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None

w = t.tensor([4.0], requires_grad=True)
opt = InferenceSGD([w], 0.1)
((w - 1) ** 2).backward()
opt.step()
print(float(w))


def _test():
    w = t.tensor([4.0], requires_grad=True)
    opt = InferenceSGD([w], 0.1)
    # one step must NOT raise the leaf-in-place RuntimeError
    ((w - 1) ** 2).backward()
    before = float(w)
    opt.step()
    after = float(w)
    # grad at w=4 is 2*(4-1)=6, so update is -0.1*6 = -0.6 -> 3.4 (independent closed form)
    assert abs(after - 3.4) < 1e-5, after
    assert after < before


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(3)

class InferenceSGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    @t.inference_mode()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None

w = t.tensor([4.0], requires_grad=True)
opt = InferenceSGD([w], 0.1)
((w - 1) ** 2).backward()
opt.step()
print(float(w))
```
</details>